In [ ]:
! pip install numpy matplotlib scipy 

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import spectrogram, stft
import os
import warnings
warnings.filterwarnings('ignore')

# =======================
# 1. CREATE DIRECTORIES
# =======================

# Create main directory
main_dir = "modulation_plots"
os.makedirs(main_dir, exist_ok=True)

# Create subdirectories for each modulation
mod_names = [
    'NM', 'LFM', 'DLFM', 'MLFM', 'EQFM', 'SFM',
    'BFSK', 'QFSK', 'BPSK', 'Frank', 'P1', 'P2', 'P3', 'P4', 'LFM-BPSK'
]

for mod in mod_names:
    mod_dir = os.path.join(main_dir, mod)
    os.makedirs(mod_dir, exist_ok=True)

    # Create subdirectories for different plot types
    os.makedirs(os.path.join(mod_dir, "time_domain"), exist_ok=True)
    os.makedirs(os.path.join(mod_dir, "frequency_domain"), exist_ok=True)
    os.makedirs(os.path.join(mod_dir, "spectrograms"), exist_ok=True)
    os.makedirs(os.path.join(mod_dir, "phase_freq"), exist_ok=True)
    os.makedirs(os.path.join(mod_dir, "zoomed_views"), exist_ok=True)  # NEW
    os.makedirs(os.path.join(mod_dir, "summary"), exist_ok=True)

print("✅ Directory structure created:")
print(f"📁 Main directory: {main_dir}")
for mod in mod_names:
    print(f"   └── {mod}/")
    print(f"       ├── time_domain/")
    print(f"       ├── frequency_domain/")
    print(f"       ├── spectrograms/")
    print(f"       ├── phase_freq/")
    print(f"       ├── zoomed_views/")  # NEW
    print(f"       └── summary/")

# =======================
# 2. PARAMETERS
# =======================
fs = 100e6          # Sampling frequency (100 MHz)
PW = 20.48e-6       # Pulse width (20.48 µs)
t = np.linspace(0, PW, int(fs * PW), endpoint=False)  # Time vector
N = len(t)

# Modulation parameters (example values)
f0 = 25e6           # Center frequency (25 MHz)
delta_f = 20e6      # Bandwidth (20 MHz)
f_min = f0 - delta_f/2

# =======================
# 3. MODULATION FUNCTIONS
# =======================

def generate_nm(t, f0):
    """NM: No Modulation (constant frequency)"""
    phi = 2 * np.pi * f0 * t
    return np.exp(1j * phi)

def generate_lfm(t, f_min, delta_f, PW, direction='up'):
    """LFM: Linear Frequency Modulation (chirp)"""
    if direction == 'up':
        f_t = f_min + (delta_f / PW) * t
    else:
        f_t = f_min + delta_f - (delta_f / PW) * t
    phi = 2 * np.pi * np.cumsum(f_t) * (t[1] - t[0])
    return np.exp(1j * phi)

def generate_dlfm(t, f_min, delta_f, PW):
    """DLFM: Dual-rate LFM (two chirp segments)"""
    t_mid = PW / 2
    idx = t < t_mid
    f_t = np.zeros_like(t)
    f_t[idx] = f_min + (delta_f / t_mid) * t[idx]
    f_t[~idx] = f_min + delta_f - (delta_f / (PW - t_mid)) * (t[~idx] - t_mid)
    phi = 2 * np.pi * np.cumsum(f_t) * (t[1] - t[0])
    return np.exp(1j * phi)

def generate_mlfm(t, f_min, delta_f, PW, r=0.2):
    """MLFM: Multi-rate LFM (multiple chirp segments)"""
    t1 = r * PW
    idx1 = t < t1
    f_t = np.zeros_like(t)
    f_t[idx1] = f_min + (delta_f / t1) * t[idx1]
    idx2 = t >= t1
    f_t[idx2] = f_min + delta_f - (delta_f / (PW - t1)) * (t[idx2] - t1)
    phi = 2 * np.pi * np.cumsum(f_t) * (t[1] - t[0])
    return np.exp(1j * phi)

def generate_eqfm(t, f_min, delta_f, PW, n_steps=5):
    """EQFM: Equipartition FM (stepped frequency)"""
    step_duration = PW / n_steps
    f_t = np.zeros_like(t)
    for i in range(n_steps):
        idx = (t >= i*step_duration) & (t < (i+1)*step_duration)
        f_t[idx] = f_min + (i / (n_steps-1)) * delta_f
    phi = 2 * np.pi * np.cumsum(f_t) * (t[1] - t[0])
    return np.exp(1j * phi)

def generate_sfm(t, f_min, delta_f, PW, f_sfm, phi_sfm=0):
    """SFM: Sinusoidal Frequency Modulation"""
    f_t = f_min + delta_f/2 + (delta_f/2) * np.sin(2*np.pi*f_sfm*t + phi_sfm)
    phi = 2 * np.pi * np.cumsum(f_t) * (t[1] - t[0])
    return np.exp(1j * phi)

def generate_bfsk(t, f0, delta_f, PW, code='normal'):
    """BFSK: Binary Frequency Shift Keying (2 frequencies)"""
    f1 = f0 - delta_f/2
    f2 = f0 + delta_f/2
    barker = np.array([1, 1, 1, 1, 1, -1, -1, 1, 1, -1, 1, -1, 1])
    if code == 'inverted':
        barker = -barker
    symbol_duration = PW / len(barker)
    f_t = np.zeros_like(t)
    for i, bit in enumerate(barker):
        idx = (t >= i*symbol_duration) & (t < (i+1)*symbol_duration)
        f_t[idx] = f1 if bit == 1 else f2
    phi = 2 * np.pi * np.cumsum(f_t) * (t[1] - t[0])
    return np.exp(1j * phi)

def generate_qfsk(t, f0, delta_f, PW):
    """QFSK: Quadrature Frequency Shift Keying (4 frequencies)"""
    frank_code = np.array([0, 0, 0, 0,
                           0, 1, 2, 3,
                           0, 2, 0, 2,
                           0, 3, 2, 1])
    N_sym = 4
    freqs = [f0 - 3*delta_f/4, f0 - delta_f/4, f0 + delta_f/4, f0 + 3*delta_f/4]
    symbol_duration = PW / len(frank_code)
    f_t = np.zeros_like(t)
    for i, symbol in enumerate(frank_code):
        idx = (t >= i*symbol_duration) & (t < (i+1)*symbol_duration)
        f_t[idx] = freqs[symbol]
    phi = 2 * np.pi * np.cumsum(f_t) * (t[1] - t[0])
    return np.exp(1j * phi)

def generate_bpsk(t, f0, PW, code='normal'):
    """BPSK: Binary Phase Shift Keying"""
    barker = np.array([1, 1, 1, 1, 1, -1, -1, 1, 1, -1, 1, -1, 1])
    if code == 'inverted':
        barker = -barker
    symbol_duration = PW / len(barker)
    phi_t = np.zeros_like(t)
    for i, bit in enumerate(barker):
        idx = (t >= i*symbol_duration) & (t < (i+1)*symbol_duration)
        phi_t[idx] = 0 if bit == 1 else np.pi
    carrier = 2 * np.pi * f0 * t
    return np.exp(1j * (carrier + phi_t))

def generate_frank(t, f0, PW, N_order=8):
    """Frank code: Polyphase code (stepped phase)"""
    n = int(np.sqrt(N_order))
    phase_matrix = np.zeros((n, n))
    for p in range(n):
        for q in range(n):
            phase_matrix[p, q] = 2*np.pi * p * q / n
    phases = phase_matrix.flatten()
    symbol_duration = PW / len(phases)
    phi_t = np.zeros_like(t)
    for i, phase in enumerate(phases):
        idx = (t >= i*symbol_duration) & (t < (i+1)*symbol_duration)
        phi_t[idx] = phase
    carrier = 2 * np.pi * f0 * t
    return np.exp(1j * (carrier + phi_t))

def generate_px(t, f0, PW, code_type='P1', N_order=8):
    """P1, P2, P3, P4: Polyphase codes"""
    n = N_order
    phases = np.zeros(n)
    if code_type == 'P1':
        for k in range(n):
            phases[k] = -np.pi * (n-1-2*k)**2 / n
    elif code_type == 'P2':
        for k in range(n):
            phases[k] = -np.pi * k**2 / n
    elif code_type == 'P3':
        for k in range(n):
            phases[k] = -np.pi * (k-1)**2 / n
    elif code_type == 'P4':
        for k in range(n):
            phases[k] = -np.pi * (k-1)**2 / n + np.pi * (k-1)
    else:
        raise ValueError("code_type must be P1, P2, P3, or P4")

    symbol_duration = PW / n
    phi_t = np.zeros_like(t)
    for i, phase in enumerate(phases):
        idx = (t >= i*symbol_duration) & (t < (i+1)*symbol_duration)
        phi_t[idx] = phase
    carrier = 2 * np.pi * f0 * t
    return np.exp(1j * (carrier + phi_t))

def generate_lfm_bpsk(t, f_min, delta_f, PW, code='normal'):
    """LFM-BPSK: Hybrid LFM + BPSK"""
    f_t = f_min + (delta_f / PW) * t
    phi_lfm = 2 * np.pi * np.cumsum(f_t) * (t[1] - t[0])

    barker = np.array([1, 1, 1, 1, 1, -1, -1, 1, 1, -1, 1, -1, 1])
    if code == 'inverted':
        barker = -barker
    symbol_duration = PW / len(barker)
    phi_bpsk = np.zeros_like(t)
    for i, bit in enumerate(barker):
        idx = (t >= i*symbol_duration) & (t < (i+1)*symbol_duration)
        phi_bpsk[idx] = 0 if bit == 1 else np.pi

    return np.exp(1j * (phi_lfm + phi_bpsk))

# =======================
# 4. GENERATE ALL SIGNALS
# =======================

signals = {}

print("\n" + "="*60)
print("GENERATING SIGNALS FOR 15 MODULATION TYPES")
print("="*60)

# Generate each modulation
signals['NM'] = generate_nm(t, f0)
print("✅ NM generated")

signals['LFM'] = generate_lfm(t, f_min, delta_f, PW, direction='up')
print("✅ LFM generated")

signals['DLFM'] = generate_dlfm(t, f_min, delta_f, PW)
print("✅ DLFM generated")

signals['MLFM'] = generate_mlfm(t, f_min, delta_f, PW, r=0.2)
print("✅ MLFM generated")

signals['EQFM'] = generate_eqfm(t, f_min, delta_f, PW, n_steps=5)
print("✅ EQFM generated")

signals['SFM'] = generate_sfm(t, f_min, delta_f, PW, f_sfm=2/PW, phi_sfm=0)
print("✅ SFM generated")

signals['BFSK'] = generate_bfsk(t, f0, delta_f, PW, code='normal')
print("✅ BFSK generated")

signals['QFSK'] = generate_qfsk(t, f0, delta_f, PW)
print("✅ QFSK generated")

signals['BPSK'] = generate_bpsk(t, f0, PW, code='normal')
print("✅ BPSK generated")

signals['Frank'] = generate_frank(t, f0, PW, N_order=8)
print("✅ Frank generated")

signals['P1'] = generate_px(t, f0, PW, 'P1', 8)
print("✅ P1 generated")

signals['P2'] = generate_px(t, f0, PW, 'P2', 8)
print("✅ P2 generated")

signals['P3'] = generate_px(t, f0, PW, 'P3', 8)
print("✅ P3 generated")

signals['P4'] = generate_px(t, f0, PW, 'P4', 8)
print("✅ P4 generated")

signals['LFM-BPSK'] = generate_lfm_bpsk(t, f_min, delta_f, PW, code='normal')
print("✅ LFM-BPSK generated")

print("\n" + "="*60)
print("ALL SIGNALS GENERATED SUCCESSFULLY!")
print("="*60)

# =======================
# 5. VISUALIZATION FUNCTIONS (UPDATED)
# =======================

def save_time_domain_plots(signal, t, name, save_dir):
    """Save time domain plots"""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6))

    # Real part
    ax1.plot(t[:500] * 1e6, np.real(signal)[:500], linewidth=1.5, color='b')
    ax1.set_xlabel('Time (µs)')
    ax1.set_ylabel('Amplitude')
    ax1.set_title(f'{name} - Real Part (First 500 samples)')
    ax1.grid(True, alpha=0.3)

    # Imaginary part
    ax2.plot(t[:500] * 1e6, np.imag(signal)[:500], linewidth=1.5, color='r')
    ax2.set_xlabel('Time (µs)')
    ax2.set_ylabel('Amplitude')
    ax2.set_title(f'{name} - Imaginary Part (First 500 samples)')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "time_domain", f"{name}_time_domain.png"), dpi=150, bbox_inches='tight')
    plt.close()

    # Save raw data plot
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(t * 1e6, np.real(signal), linewidth=0.5, alpha=0.7, color='b', label='Real')
    ax.plot(t * 1e6, np.imag(signal), linewidth=0.5, alpha=0.7, color='r', label='Imag')
    ax.set_xlabel('Time (µs)')
    ax.set_ylabel('Amplitude')
    ax.set_title(f'{name} - Complete Signal')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.savefig(os.path.join(save_dir, "time_domain", f"{name}_complete_signal.png"), dpi=150, bbox_inches='tight')
    plt.close()

def save_frequency_domain_plots(signal, t, name, fs, save_dir):
    """Save frequency domain plots"""
    freq = np.fft.fftfreq(len(signal), 1/fs)
    spectrum = np.abs(np.fft.fft(signal))
    idx = freq >= 0

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(freq[idx]/1e6, spectrum[idx], linewidth=1.5, color='g')
    ax.set_xlabel('Frequency (MHz)')
    ax.set_ylabel('Magnitude')
    ax.set_title(f'{name} - Frequency Spectrum')
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 50])
    plt.savefig(os.path.join(save_dir, "frequency_domain", f"{name}_spectrum.png"), dpi=150, bbox_inches='tight')
    plt.close()

    # Log scale plot
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(freq[idx]/1e6, 20*np.log10(spectrum[idx] + 1e-10), linewidth=1.5, color='purple')
    ax.set_xlabel('Frequency (MHz)')
    ax.set_ylabel('Magnitude (dB)')
    ax.set_title(f'{name} - Frequency Spectrum (dB scale)')
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 50])
    plt.savefig(os.path.join(save_dir, "frequency_domain", f"{name}_spectrum_dB.png"), dpi=150, bbox_inches='tight')
    plt.close()

def save_spectrogram_plots(signal, t, name, fs, save_dir):
    """Save spectrogram plots - FIXED USING STFT"""
    # Use STFT with real part only (as in working code)
    signal_real = np.real(signal)
    nperseg = 256
    noverlap = 128
    f, t_spec, Zxx = stft(signal_real, fs=fs, nperseg=nperseg, noverlap=noverlap)
    
    # Compute magnitude and convert to dB (20*log10 for amplitude)
    Sxx_mag = np.abs(Zxx)
    Sxx_db = 20 * np.log10(Sxx_mag + 1e-10)
    
    # Find reasonable color limits
    vmax = np.max(Sxx_db)
    vmin = vmax - 40  # Show 40 dB dynamic range

    # Standard spectrogram
    fig, ax = plt.subplots(figsize=(10, 6))
    im = ax.pcolormesh(t_spec * 1e6, f/1e6, Sxx_db, shading='gouraud', cmap='viridis', vmin=vmin, vmax=vmax)
    ax.set_xlabel('Time (µs)')
    ax.set_ylabel('Frequency (MHz)')
    ax.set_title(f'{name} - Spectrogram (STFT)')
    ax.set_ylim([0, 50])
    plt.colorbar(im, ax=ax, label='Power (dB)')
    plt.savefig(os.path.join(save_dir, "spectrograms", f"{name}_spectrogram_stft.png"), dpi=150, bbox_inches='tight')
    plt.close()

    # Also save traditional spectrogram for comparison
    f_spec, t_spec_spec, Sxx_spec = spectrogram(signal_real, fs=fs, nperseg=256, noverlap=128)
    Sxx_db_spec = 10 * np.log10(np.abs(Sxx_spec)**2 + 1e-10)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    im = ax.pcolormesh(t_spec_spec * 1e6, f_spec/1e6, Sxx_db_spec, shading='gouraud', cmap='jet')
    ax.set_xlabel('Time (µs)')
    ax.set_ylabel('Frequency (MHz)')
    ax.set_title(f'{name} - Spectrogram (Traditional)')
    ax.set_ylim([0, 50])
    plt.colorbar(im, ax=ax, label='Power (dB)')
    plt.savefig(os.path.join(save_dir, "spectrograms", f"{name}_spectrogram_traditional.png"), dpi=150, bbox_inches='tight')
    plt.close()

def save_phase_frequency_plots(signal, t, name, fs, save_dir):
    """Save phase and instantaneous frequency plots"""
    phase = np.unwrap(np.angle(signal))
    instantaneous_freq = np.diff(phase) / (2*np.pi) * fs

    # Instantaneous frequency
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(t[:-1]*1e6, instantaneous_freq/1e6, linewidth=1.5, color='purple')
    ax.set_xlabel('Time (µs)')
    ax.set_ylabel('Frequency (MHz)')
    ax.set_title(f'{name} - Instantaneous Frequency')
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 50])
    plt.savefig(os.path.join(save_dir, "phase_freq", f"{name}_instantaneous_freq.png"), dpi=150, bbox_inches='tight')
    plt.close()

    # Phase plot
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(t*1e6, phase, linewidth=1.5, color='orange')
    ax.set_xlabel('Time (µs)')
    ax.set_ylabel('Phase (rad)')
    ax.set_title(f'{name} - Unwrapped Phase')
    ax.grid(True, alpha=0.3)
    plt.savefig(os.path.join(save_dir, "phase_freq", f"{name}_phase.png"), dpi=150, bbox_inches='tight')
    plt.close()

def save_zoomed_views(signal, t, name, fs, save_dir):
    """Save zoomed views of the signal - NEW FUNCTION"""
    # Zoom duration: first 0.2 µs (200 nanoseconds)
    zoom_duration = 0.2e-6
    zoom_samples = int(zoom_duration * fs)
    zoom_samples = min(zoom_samples, len(t)//5)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'{name} - Zoomed Views (First {zoom_duration*1e6:.2f} µs)', fontsize=14, fontweight='bold')
    
    # Zoomed Real part
    axes[0].plot(t[:zoom_samples] * 1e6, np.real(signal)[:zoom_samples], 
                  linewidth=2, color='b', marker='o', markersize=3)
    axes[0].set_xlabel('Time (µs)')
    axes[0].set_ylabel('Amplitude')
    axes[0].set_title('Real Part (Zoomed)')
    axes[0].grid(True, alpha=0.3)
    
    # Zoomed Imaginary part
    axes[1].plot(t[:zoom_samples] * 1e6, np.imag(signal)[:zoom_samples], 
                  linewidth=2, color='r', marker='s', markersize=3)
    axes[1].set_xlabel('Time (µs)')
    axes[1].set_ylabel('Amplitude')
    axes[1].set_title('Imaginary Part (Zoomed)')
    axes[1].grid(True, alpha=0.3)
    
    # Zoomed IQ plot
    axes[2].scatter(np.real(signal[:zoom_samples]), np.imag(signal[:zoom_samples]), 
                   c=np.arange(zoom_samples), cmap='viridis', s=10, alpha=0.7)
    axes[2].set_xlabel('I (Real)')
    axes[2].set_ylabel('Q (Imaginary)')
    axes[2].set_title('IQ Constellation (Zoomed)')
    axes[2].grid(True, alpha=0.3)
    axes[2].set_aspect('equal')
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "zoomed_views", f"{name}_zoomed_views.png"), dpi=150, bbox_inches='tight')
    plt.close()
    
    # Also save super-zoomed view (first 0.05 µs)
    super_zoom_duration = 0.05e-6
    super_zoom_samples = int(super_zoom_duration * fs)
    super_zoom_samples = min(super_zoom_samples, len(t)//20)
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(t[:super_zoom_samples] * 1e6, np.real(signal)[:super_zoom_samples], 
            'b-o', markersize=4, linewidth=1.5, label='Real')
    ax.plot(t[:super_zoom_samples] * 1e6, np.imag(signal)[:super_zoom_samples], 
            'r-s', markersize=4, linewidth=1.5, label='Imag')
    ax.set_xlabel('Time (µs)')
    ax.set_ylabel('Amplitude')
    ax.set_title(f'{name} - Super Zoomed (First {super_zoom_duration*1e6:.2f} µs)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "zoomed_views", f"{name}_super_zoomed.png"), dpi=150, bbox_inches='tight')
    plt.close()

def save_summary_plot(signal, t, name, fs, save_dir):
    """Save a comprehensive summary plot - UPDATED"""
    fig, axes = plt.subplots(3, 3, figsize=(18, 14))  # Increased to 3x3 for more plots
    fig.suptitle(f'{name} Modulation - Complete Analysis', fontsize=16, fontweight='bold')

    # 1. Real part (500 samples)
    axes[0, 0].plot(t[:500] * 1e6, np.real(signal)[:500], 'b-', linewidth=1.5)
    axes[0, 0].set_xlabel('Time (µs)')
    axes[0, 0].set_ylabel('Amplitude')
    axes[0, 0].set_title('Real Part (500 samples)')
    axes[0, 0].grid(True, alpha=0.3)

    # 2. Imaginary part (500 samples)
    axes[0, 1].plot(t[:500] * 1e6, np.imag(signal)[:500], 'r-', linewidth=1.5)
    axes[0, 1].set_xlabel('Time (µs)')
    axes[0, 1].set_ylabel('Amplitude')
    axes[0, 1].set_title('Imaginary Part (500 samples)')
    axes[0, 1].grid(True, alpha=0.3)

    # 3. Frequency spectrum
    freq = np.fft.fftfreq(len(signal), 1/fs)
    spectrum = np.abs(np.fft.fft(signal))
    idx = freq >= 0
    axes[0, 2].plot(freq[idx]/1e6, 20*np.log10(spectrum[idx] + 1e-10), 'g-', linewidth=1.5)
    axes[0, 2].set_xlabel('Frequency (MHz)')
    axes[0, 2].set_ylabel('Magnitude (dB)')
    axes[0, 2].set_title('Frequency Spectrum')
    axes[0, 2].grid(True, alpha=0.3)
    axes[0, 2].set_xlim([0, 50])

    # 4. Instantaneous frequency
    phase = np.unwrap(np.angle(signal))
    instantaneous_freq = np.diff(phase) / (2*np.pi) * fs
    axes[1, 0].plot(t[:-1]*1e6, instantaneous_freq/1e6, 'purple', linewidth=1.5)
    axes[1, 0].set_xlabel('Time (µs)')
    axes[1, 0].set_ylabel('Frequency (MHz)')
    axes[1, 0].set_title('Instantaneous Frequency')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].set_ylim([0, 50])

    # 5. Phase
    axes[1, 1].plot(t*1e6, phase, 'orange', linewidth=1.5)
    axes[1, 1].set_xlabel('Time (µs)')
    axes[1, 1].set_ylabel('Phase (rad)')
    axes[1, 1].set_title('Unwrapped Phase')
    axes[1, 1].grid(True, alpha=0.3)

    # 6. IQ plot (constellation)
    axes[1, 2].scatter(np.real(signal[::50]), np.imag(signal[::50]),
                       c=np.arange(len(signal[::50])), cmap='viridis', s=10, alpha=0.7)
    axes[1, 2].set_xlabel('I (Real)')
    axes[1, 2].set_ylabel('Q (Imaginary)')
    axes[1, 2].set_title('IQ Constellation')
    axes[1, 2].grid(True, alpha=0.3)
    axes[1, 2].set_aspect('equal')

    # 7. FIXED SPECTROGRAM using STFT
    signal_real = np.real(signal)
    nperseg = 256
    noverlap = 128
    f, t_spec, Zxx = stft(signal_real, fs=fs, nperseg=nperseg, noverlap=noverlap)
    Sxx_mag = np.abs(Zxx)
    Sxx_db = 20 * np.log10(Sxx_mag + 1e-10)
    vmax = np.max(Sxx_db)
    vmin = vmax - 40
    
    im = axes[2, 0].pcolormesh(t_spec * 1e6, f/1e6, Sxx_db, 
                               shading='gouraud', cmap='viridis', vmin=vmin, vmax=vmax)
    axes[2, 0].set_xlabel('Time (µs)')
    axes[2, 0].set_ylabel('Frequency (MHz)')
    axes[2, 0].set_title('Spectrogram (STFT)')
    axes[2, 0].set_ylim([0, 50])
    plt.colorbar(im, ax=axes[2, 0], label='Power (dB)')

    # 8. Zoomed real part (first 0.2 µs)
    zoom_duration = 0.2e-6
    zoom_samples = int(zoom_duration * fs)
    zoom_samples = min(zoom_samples, len(t)//5)
    axes[2, 1].plot(t[:zoom_samples] * 1e6, np.real(signal)[:zoom_samples], 
                    'b-o', markersize=3, linewidth=1.5)
    axes[2, 1].set_xlabel('Time (µs)')
    axes[2, 1].set_ylabel('Amplitude')
    axes[2, 1].set_title(f'Real Part (First {zoom_duration*1e6:.2f} µs)')
    axes[2, 1].grid(True, alpha=0.3)

    # 9. Zoomed imaginary part (first 0.2 µs)
    axes[2, 2].plot(t[:zoom_samples] * 1e6, np.imag(signal)[:zoom_samples], 
                    'r-s', markersize=3, linewidth=1.5)
    axes[2, 2].set_xlabel('Time (µs)')
    axes[2, 2].set_ylabel('Amplitude')
    axes[2, 2].set_title(f'Imag Part (First {zoom_duration*1e6:.2f} µs)')
    axes[2, 2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "summary", f"{name}_summary.png"), dpi=150, bbox_inches='tight')
    plt.close()

# =======================
# 6. SAVE ALL PLOTS
# =======================

print("\n" + "="*60)
print("SAVING PLOTS TO DIRECTORIES")
print("="*60)

for i, name in enumerate(mod_names):
    print(f"\n📊 Processing {name}...")
    signal = signals[name]
    mod_dir = os.path.join(main_dir, name)

    # Save all plots
    save_time_domain_plots(signal, t, name, mod_dir)
    save_frequency_domain_plots(signal, t, name, fs, mod_dir)
    save_spectrogram_plots(signal, t, name, fs, mod_dir)  # FIXED
    save_phase_frequency_plots(signal, t, name, fs, mod_dir)
    save_zoomed_views(signal, t, name, fs, mod_dir)  # NEW
    save_summary_plot(signal, t, name, fs, mod_dir)  # UPDATED

    print(f"   ✅ Saved 6 sets of plots in {mod_dir}/")

# =======================
# 7. CREATE COMPARISON GRIDS - UPDATED
# =======================

print("\n" + "="*60)
print("CREATING COMPARISON GRIDS (WITH FIXED SPECTROGRAMS)")
print("="*60)

# Create comparison directory
compare_dir = os.path.join(main_dir, "comparisons")
os.makedirs(compare_dir, exist_ok=True)

# 1. All spectrograms comparison using STFT (FIXED)
fig, axes = plt.subplots(5, 3, figsize=(18, 20))
axes = axes.flatten()

# First compute all spectrograms to find global color limits
all_Sxx_db = []
for name in mod_names:
    signal_real = np.real(signals[name])
    nperseg = 256
    noverlap = 128
    f, t_spec, Zxx = stft(signal_real, fs=fs, nperseg=nperseg, noverlap=noverlap)
    Sxx_mag = np.abs(Zxx)
    Sxx_db = 20 * np.log10(Sxx_mag + 1e-10)
    all_Sxx_db.append(Sxx_db)

# Find global min/max for consistent coloring
all_data = np.concatenate([sxx.flatten() for sxx in all_Sxx_db])
vmax = np.max(all_data)
vmin = vmax - 40  # Show 40 dB dynamic range

for i, name in enumerate(mod_names):
    signal_real = np.real(signals[name])
    nperseg = 256
    noverlap = 128
    f, t_spec, Zxx = stft(signal_real, fs=fs, nperseg=nperseg, noverlap=noverlap)
    Sxx_mag = np.abs(Zxx)
    Sxx_db = 20 * np.log10(Sxx_mag + 1e-10)
    
    im = axes[i].pcolormesh(t_spec * 1e6, f/1e6, Sxx_db, 
                           shading='gouraud', cmap='viridis', vmin=vmin, vmax=vmax)
    axes[i].set_title(f'{name}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Time (µs)')
    axes[i].set_ylabel('Freq (MHz)')
    axes[i].set_ylim([0, 50])
    axes[i].set_aspect('auto')

# Remove unused subplots
for i in range(len(mod_names), len(axes)):
    fig.delaxes(axes[i])

plt.suptitle('Comparison of All 15 Modulation Types (Spectrograms - STFT)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(compare_dir, "all_spectrograms_stft.png"), dpi=150, bbox_inches='tight')
plt.close()
print("✅ Saved: all_spectrograms_stft.png")

# 2. Instantaneous frequency comparison
fig, axes = plt.subplots(5, 3, figsize=(18, 20))
axes = axes.flatten()

for i, name in enumerate(mod_names):
    signal = signals[name]
    phase = np.unwrap(np.angle(signal))
    instantaneous_freq = np.diff(phase) / (2*np.pi) * fs

    axes[i].plot(t[:-1]*1e6, instantaneous_freq/1e6, linewidth=1.5, color='purple')
    axes[i].set_title(f'{name}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Time (µs)')
    axes[i].set_ylabel('Freq (MHz)')
    axes[i].set_ylim([0, 50])
    axes[i].grid(True, alpha=0.3)

for i in range(len(mod_names), len(axes)):
    fig.delaxes(axes[i])

plt.suptitle('Comparison of Instantaneous Frequency', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(compare_dir, "instantaneous_frequencies.png"), dpi=150, bbox_inches='tight')
plt.close()
print("✅ Saved: instantaneous_frequencies.png")

# 3. Frequency spectra comparison
fig, axes = plt.subplots(5, 3, figsize=(18, 20))
axes = axes.flatten()

for i, name in enumerate(mod_names):
    signal = signals[name]
    freq = np.fft.fftfreq(len(signal), 1/fs)
    spectrum = np.abs(np.fft.fft(signal))
    idx = freq >= 0

    axes[i].plot(freq[idx]/1e6, 20*np.log10(spectrum[idx] + 1e-10), linewidth=1.5, color='g')
    axes[i].set_title(f'{name}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Frequency (MHz)')
    axes[i].set_ylabel('Magnitude (dB)')
    axes[i].set_xlim([0, 50])
    axes[i].grid(True, alpha=0.3)

for i in range(len(mod_names), len(axes)):
    fig.delaxes(axes[i])

plt.suptitle('Comparison of Frequency Spectra', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(compare_dir, "frequency_spectra.png"), dpi=150, bbox_inches='tight')
plt.close()
print("✅ Saved: frequency_spectra.png")

# 4. Zoomed views comparison (first 0.2 µs)
fig, axes = plt.subplots(5, 3, figsize=(18, 20))
axes = axes.flatten()

zoom_duration = 0.2e-6
zoom_samples = int(zoom_duration * fs)

for i, name in enumerate(mod_names):
    signal = signals[name]
    zoom_samples = min(zoom_samples, len(t)//5)
    
    axes[i].plot(t[:zoom_samples] * 1e6, np.real(signal)[:zoom_samples], 
                 'b-', linewidth=1.5, label='Real')
    axes[i].plot(t[:zoom_samples] * 1e6, np.imag(signal)[:zoom_samples], 
                 'r-', linewidth=1.5, label='Imag')
    axes[i].set_title(f'{name}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Time (µs)')
    axes[i].set_ylabel('Amplitude')
    axes[i].grid(True, alpha=0.3)
    if i == 0:
        axes[i].legend()

for i in range(len(mod_names), len(axes)):
    fig.delaxes(axes[i])

plt.suptitle(f'Comparison of Zoomed Views (First {zoom_duration*1e6:.2f} µs)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(compare_dir, "zoomed_views_comparison.png"), dpi=150, bbox_inches='tight')
plt.close()
print("✅ Saved: zoomed_views_comparison.png")

# =======================
# 8. CREATE HTML SUMMARY - UPDATED
# =======================

print("\n" + "="*60)
print("GENERATING HTML SUMMARY")
print("="*60)

html_content = """
<!DOCTYPE html>
<html>
<head>
    <title>Radar Signal Modulation Visualization</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 40px; background: #f5f5f5; }
        .container { max-width: 1200px; margin: 0 auto; background: white; padding: 30px; border-radius: 10px; }
        h1 { color: #2c3e50; border-bottom: 3px solid #3498db; padding-bottom: 10px; }
        h2 { color: #34495e; margin-top: 30px; }
        .mod-grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 20px; margin: 20px 0; }
        .mod-card { background: #ecf0f1; border-radius: 8px; padding: 15px; transition: transform 0.3s; }
        .mod-card:hover { transform: translateY(-5px); box-shadow: 0 5px 15px rgba(0,0,0,0.1); }
        .mod-title { font-weight: bold; color: #2c3e50; margin-bottom: 10px; }
        .plot-links { display: flex; flex-direction: column; gap: 5px; }
        .plot-link { color: #2980b9; text-decoration: none; padding: 5px; border-radius: 4px; }
        .plot-link:hover { background: #3498db; color: white; }
        .summary-img { max-width: 100%; border-radius: 5px; margin: 10px 0; }
        .comparison-grid { display: grid; grid-template-columns: repeat(2, 1fr); gap: 20px; }
        .comparison-card { text-align: center; }
        .improvements { background: #e8f4fc; padding: 15px; border-radius: 8px; margin: 20px 0; }
        .improvement-item { color: #2c3e50; margin: 5px 0; }
    </style>
</head>
<body>
    <div class="container">
        <h1>📡 Radar Signal Modulation Visualization</h1>
        <p>Generated: """ + str(np.datetime64('now')) + """</p>
        
        <div class="improvements">
            <h3>✨ Key Improvements in This Version:</h3>
            <div class="improvement-item">✅ Fixed spectrograms using STFT (20*log10 scaling)</div>
            <div class="improvement-item">✅ Added zoomed time plots (first 0.2 µs)</div>
            <div class="improvement-item">✅ Added super-zoomed plots (first 0.05 µs)</div>
            <div class="improvement-item">✅ Improved color scaling for consistent visualization</div>
            <div class="improvement-item">✅ Added IQ constellation plots in zoomed views</div>
        </div>

        <h2>📊 Individual Modulation Analysis</h2>
        <div class="mod-grid">
"""

# Add modulation cards
for name in mod_names:
    mod_dir = os.path.join(main_dir, name)
    summary_path = os.path.join(mod_dir, "summary", f"{name}_summary.png")
    html_content += f"""
            <div class="mod-card">
                <div class="mod-title">{name}</div>
                <img src="{summary_path}" alt="{name} Summary" class="summary-img">
                <div class="plot-links">
                    <a href="{mod_dir}/time_domain/{name}_time_domain.png" class="plot-link">📈 Time Domain</a>
                    <a href="{mod_dir}/frequency_domain/{name}_spectrum.png" class="plot-link">📊 Frequency Spectrum</a>
                    <a href="{mod_dir}/spectrograms/{name}_spectrogram_stft.png" class="plot-link">🎨 Spectrogram (STFT)</a>
                    <a href="{mod_dir}/phase_freq/{name}_instantaneous_freq.png" class="plot-link">📐 Instantaneous Frequency</a>
                    <a href="{mod_dir}/zoomed_views/{name}_zoomed_views.png" class="plot-link">🔍 Zoomed Views</a>
                    <a href="{mod_dir}/summary/{name}_summary.png" class="plot-link">📋 Complete Summary</a>
                </div>
            </div>
"""

html_content += """
        </div>

        <h2>🔍 Comparison Plots</h2>
        <div class="comparison-grid">
            <div class="comparison-card">
                <h3>All Spectrograms (STFT)</h3>
                <a href="comparisons/all_spectrograms_stft.png">
                    <img src="comparisons/all_spectrograms_stft.png" alt="All Spectrograms" style="width: 100%; border-radius: 5px;">
                </a>
            </div>
            <div class="comparison-card">
                <h3>Instantaneous Frequencies</h3>
                <a href="comparisons/instantaneous_frequencies.png">
                    <img src="comparisons/instantaneous_frequencies.png" alt="Instantaneous Frequencies" style="width: 100%; border-radius: 5px;">
                </a>
            </div>
            <div class="comparison-card">
                <h3>Frequency Spectra</h3>
                <a href="comparisons/frequency_spectra.png">
                    <img src="comparisons/frequency_spectra.png" alt="Frequency Spectra" style="width: 100%; border-radius: 5px;">
                </a>
            </div>
            <div class="comparison-card">
                <h3>Zoomed Views</h3>
                <a href="comparisons/zoomed_views_comparison.png">
                    <img src="comparisons/zoomed_views_comparison.png" alt="Zoomed Views" style="width: 100%; border-radius: 5px;">
                </a>
            </div>
        </div>

        <h2>📁 Directory Structure</h2>
        <pre>
/content/modulation_plots/
├── NM/
│   ├── time_domain/
│   ├── frequency_domain/
│   ├── spectrograms/          # Contains STFT and traditional spectrograms
│   ├── phase_freq/
│   ├── zoomed_views/          # NEW: Contains zoomed plots
│   └── summary/
├── LFM/
│   └── ...
├── ...
├── comparisons/
└── index.html
        </pre>

        <h2>⚙️ Parameters Used</h2>
        <ul>
            <li>Sampling Frequency (f_s): 100 MHz</li>
            <li>Pulse Width (PW): 20.48 µs</li>
            <li>Center Frequency (f₀): 25 MHz</li>
            <li>Bandwidth (Δf): 20 MHz</li>
            <li>Total Samples: 2048</li>
            <li>STFT Parameters: nperseg=256, noverlap=128</li>
            <li>Zoom Duration: 0.2 µs (200 ns)</li>
            <li>Super-Zoom Duration: 0.05 µs (50 ns)</li>
        </ul>

        <footer style="margin-top: 40px; padding-top: 20px; border-top: 1px solid #ddd; color: #7f8c8d;">
            <p>Generated for: Deep Learning Models for Realistic Multicomponent Signal Modulation Classification</p>
            <p>Implementation with fixed spectrograms and enhanced visualization</p>
        </footer>
    </div>
</body>
</html>
"""

# Save HTML file
html_path = os.path.join(main_dir, "index.html")
with open(html_path, 'w') as f:
    f.write(html_content)

print(f"✅ HTML summary saved: {html_path}")

# =======================
# 9. FINAL SUMMARY
# =======================

print("\n" + "="*60)
print("VISUALIZATION COMPLETE!")
print("="*60)

# Count files
total_files = 0
for root, dirs, files in os.walk(main_dir):
    total_files += len(files)

print(f"\n📊 Summary:")
print(f"   Main directory: {main_dir}")
print(f"   Total modulation types: {len(mod_names)}")
print(f"   Total files generated: {total_files}")
print(f"   HTML report: {html_path}")

print("\n📁 Directory listing:")
for mod in mod_names:
    mod_path = os.path.join(main_dir, mod)
    mod_files = sum([len(files) for r, d, files in os.walk(mod_path)])
    print(f"   {mod:12s} - {mod_files:3d} files")

print("\n🔗 To view the HTML report in Colab:")
print(f"   from google.colab import files")
print(f"   files.view('{html_path}')")

print("\n📦 To download all plots:")
print(f"   !zip -r modulation_plots.zip {main_dir}")

print("\n" + "="*60)
print("NEXT STEP: Add noise and create dual-component signals")
print("="*60)

# Provide quick download command
print("\n💾 Quick download command:")
print(f"!zip -r /content/modulation_plots.zip {main_dir}")

✅ Directory structure created:
📁 Main directory: modulation_plots
   └── NM/
       ├── time_domain/
       ├── frequency_domain/
       ├── spectrograms/
       ├── phase_freq/
       ├── zoomed_views/
       └── summary/
   └── LFM/
       ├── time_domain/
       ├── frequency_domain/
       ├── spectrograms/
       ├── phase_freq/
       ├── zoomed_views/
       └── summary/
   └── DLFM/
       ├── time_domain/
       ├── frequency_domain/
       ├── spectrograms/
       ├── phase_freq/
       ├── zoomed_views/
       └── summary/
   └── MLFM/
       ├── time_domain/
       ├── frequency_domain/
       ├── spectrograms/
       ├── phase_freq/
       ├── zoomed_views/
       └── summary/
   └── EQFM/
       ├── time_domain/
       ├── frequency_domain/
       ├── spectrograms/
       ├── phase_freq/
       ├── zoomed_views/
       └── summary/
   └── SFM/
       ├── time_domain/
       ├── frequency_domain/
       ├── spectrograms/
       ├── phase_freq/
       ├── zoomed_views/
    